In [2]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

from src.evaluation import ranked_probability_score, rps_from_lambdas

In [4]:
# Load the cleaned dataset
results = pd.read_parquet('../data/interim/results_clean.parquet')

# Reproduce the train/eval/WC2026 split (same as previous notebooks)
model_df = results[results['date'].dt.year >= 1970].reset_index(drop=True)
wc2026_mask = (model_df['tournament'] == 'FIFA World Cup') & (model_df['date'].dt.year == 2026)
train_df = model_df[model_df['date'] < '2024-01-01']
eval_df = model_df[(model_df['date'] >= '2024-01-01') & ~wc2026_mask]
wc2026_df = model_df[wc2026_mask]

print(f'Train: {len(train_df)} | Eval: {len(eval_df)} | WC2026: {len(wc2026_df)}')

Train: 38900 | Eval: 2543 | WC2026: 79


In [5]:
model_dir = Path('../models')
cb_tuned = joblib.load(model_dir / 'cb_tuned.joblib')
lgb_tuned = joblib.load(model_dir / 'lgb_tuned.joblib')
xgb_tuned = joblib.load(model_dir / 'xgb_tuned.joblib')
hgb_tuned = joblib.load(model_dir / 'hgb_tuned.joblib')
rf_tuned = joblib.load(model_dir / 'rf_tuned.joblib')
glm_tuned = joblib.load(model_dir / 'glm_tuned.joblib')

print('6 tuned learners loaded from disk')

6 tuned learners loaded from disk
